In [1]:
from utils import get_data, final_test, confusion_mat
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
seed = 42
wine_X_tr, wine_X_val, wine_X_test, wine_y_tr, wine_y_val, wine_y_test = get_data(seed)

In [3]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.discriminant_analysis import unique_labels
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


class NN(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=[100], dropout=0.0, epochs=1000, batch_size=512, 
                 random_state=seed, **kwargs):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.kwargs = kwargs
        for key, value in kwargs.items():
            setattr(self, key, value)
        
    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X, y = check_X_y(X, y)
        self.classes_ = unique_labels(y)
        self.num_features_ = X.shape[1]
        
        y_mapped = np.searchsorted(self.classes_, y)

        layer_dims = [self.num_features_, *self.hidden_layer_sizes, len(self.classes_)]
        layers = []
        for i in range(len(layer_dims)-2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i+1]))
            layers.append(nn.ReLU())
            if self.dropout:
                layers.append(nn.Dropout(self.dropout))
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))

        self.model_ = nn.Sequential(*layers)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model_.parameters(), **self.kwargs)

        self.model_.train()
        X_tensor = torch.as_tensor(X, dtype=torch.float32)
        y_tensor = torch.as_tensor(y_mapped, dtype=torch.long)
        
        n_samples = len(X)

        for epoch in range(self.epochs):
            perm = torch.randperm(n_samples)
            X_shuffled = X_tensor[perm]
            y_shuffled = y_tensor[perm]
            
            for i in range(0, n_samples, self.batch_size):
                batch_X = X_shuffled[i : i + self.batch_size]
                batch_y = y_shuffled[i : i + self.batch_size]
                optimizer.zero_grad()
                outputs = self.model_(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

        return self

    def predict_proba(self, X):
        check_is_fitted(self)
        X = check_array(X)

        self.model_.eval()
        with torch.inference_mode():
            X_tensor = torch.as_tensor(X, dtype=torch.float32)
            outputs = self.model_(X_tensor)
            probs = torch.softmax(outputs, dim=1)

        return probs.numpy()
    
    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]


In [4]:
from sklearn.model_selection import StratifiedKFold


ensemble = StackingClassifier(
    [
        ("knn", KNeighborsClassifier(n_neighbors=1)),
        ("Logistic", LogisticRegression(solver='saga', penalty='l1', max_iter=1000, random_state=seed, C=1)),
        ("RandomForest", RandomForestClassifier(n_estimators=300, max_leaf_nodes=15000, n_jobs=-1, random_state=67)),
        ("NN", NN(
            hidden_layer_sizes=[512] * 2,
            dropout=0.06292028350373743,
            batch_size=128,
            lr=0.0021243932479146493,
            weight_decay=0.00042011473240674216
        )),
    ],
    cv=StratifiedKFold(n_splits=np.min(np.bincount(wine_y_tr)), shuffle=True, random_state=seed),
)
ensemble.fit(wine_X_tr, wine_y_tr)

/Users/alistairkeiller/Documents/cs178Final/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/alistairkeiller/Documents/cs178Final/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/alistairkeiller/Documents/cs178Final/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,estimators,"[('knn', ...), ('Logistic', ...), ...]"
,final_estimator,None
,cv,StratifiedKFo... shuffle=True)
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,n_neighbors,1
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30


In [5]:
final_test(ensemble, wine_X_test, wine_y_test)

/Users/alistairkeiller/Documents/cs178Final/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,accuracy,precision,recall,f1
0,0.651538,0.658542,0.651538,0.641197


In [6]:
confusion_mat(wine_y_test, ensemble.predict(wine_X_test), "ensemble_confusion.svg")